# Tutorial 1: Simplest Agent using Orqest

In [14]:
from pydantic import BaseModel, Field
from typing import List, Dict, Any
from orqest.agents.base_agent import BaseAgent

In [15]:
class MyState(BaseModel):
    messages: List[Dict[str, Any]] = Field(
        description="Conversation messages",
        default_factory=list
    )

    def add_message(self, role: str, content: str) -> None:
        self.messages.append({"role": role, "content": content})

class TextOutput(BaseModel):
    answer: str = Field(description="The assistant's reply text")


In [16]:
class MyAgent(BaseAgent[TextOutput]):
    def __init__(self):
        super().__init__(
            agent_name="my_agent",
            output_type=TextOutput,
            system_prompt="You are a helpful assistant to provide general information over the world.",
            retries=2
        )

    async def _run_implementation(self, state: MyState, **kwargs) -> Any:
        # Build the user prompt from the latest message
        latest = state.messages[-1] if state.messages else {"role": "system", "content": "No messages yet"}
        prompt = f"Current state, latest message: {latest}"

        # Call the LLM agent
        response = await self.agent.run(prompt, **kwargs)

        # Process and update state
        return await self._process_agent_response(response, state, **kwargs)

    async def _process_response_implementation(self, response, state: MyState, **kwargs) -> MyState:
        # response.output will be a TextOutput
        if hasattr(response, "output") and isinstance(response.output, TextOutput):
            state.add_message("assistant", response.output.answer)
        else:
            # fallback in case parsing fails
            text = getattr(response, "content", None) or str(response)
            state.add_message("assistant", text)
        return state

In [17]:
state_demo = MyState()
state_demo.add_message(role="user", content="What is the capital of France?")

agent = MyAgent()
response_state = await agent.run(state_demo)
response_state

MyState(messages=[{'role': 'user', 'content': 'What is the capital of France?'}, {'role': 'assistant', 'content': 'The capital of France is Paris.'}])